# EARL shape matching: 3 legitimate + 2 impostor users, 1 → 2 → 3 sessions

Notebook version of `experiment_earl/src/00–05`. It follows the whiteboard in `image.png`: a fixed set of users
(legitimate **L** = 3 users, impostor **I** = 2 users), and the number of sessions per user grows step by step
(1, then 2, then 3). After every step the result is recorded.

Pipeline (Balabit training files only):

1. Draw users and sessions. The sessions of step *k+1* contain the sessions of step *k*.
2. Split each session into chunks at pauses (`pause_gap_s`) and at every click.
3. Shift each chunk to the origin, scale its bounding-box diagonal to 1, resample it to `resample_n` points by arc length.
4. A chunk *matches* when a chunk of the same kind (move/drag) in the legitimate library is within banded DTW cost `dtw_tolerance`.
5. A session matches a profile when at least `match_shape_ratio` of its chunks match; an impostor user is *matched* when at least
   `match_session_ratio` of their sessions match **any single** legitimate profile.
6. `threshold = matched impostor users / total impostor users` (0 = best: impostors never look like the legitimate users).

Run with the repo's `.venv` kernel (it has numba).

In [1]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from numba import njit, prange

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()  # experiment_earl/
config = yaml.safe_load((ROOT / "config.yaml").read_text())

# What this notebook tests (everything else comes from config.yaml)
N_LEGITIMATE, N_IMPOSTOR = 3, 2
SESSION_STEPS = [1, 2, 3]
N_TRIALS = 20            # random draws of users/sessions, to see how stable each step is
SEED = config["seed"]

print({k: config[k] for k in ["pause_gap_s", "min_points", "min_length_px", "resample_n",
                              "dtw_band", "dtw_tolerance", "match_shape_ratio", "match_session_ratio"]})

{'pause_gap_s': 0.5, 'min_points': 10, 'min_length_px': 0, 'resample_n': 64, 'dtw_band': 0.1, 'dtw_tolerance': 0.02, 'match_shape_ratio': 0.3, 'match_session_ratio': 0.5}


## 1. Load a session and split it into chunks

In [2]:
INVALID_COORDINATE = 65535  # Balabit logging glitch, not a position
MOVE, DRAG = 0, 1
DATASET = ROOT / config["dataset_dir"]


def load_session(path: Path) -> pd.DataFrame:
    """Columns t (seconds from the first event), x, y, state, in time order.

    Scroll rows carry no path and glitch rows (x or y = 65535) are not positions, so both are dropped.
    The sort is stable because many Balabit timestamps tie.
    """
    df = pd.read_csv(path, usecols=["client timestamp", "button", "state", "x", "y"])
    df = df.rename(columns={"client timestamp": "t"}).dropna(subset=["t", "x", "y"])
    df = df[(df["button"] != "Scroll") & (df["x"] < INVALID_COORDINATE) & (df["y"] < INVALID_COORDINATE)]
    df = df.sort_values("t", kind="stable").reset_index(drop=True)
    df["t"] = df["t"] - df["t"].iloc[0]
    return df[["t", "x", "y", "state"]]


def path_length(chunk: np.ndarray) -> float:
    return float(np.hypot(*np.diff(chunk[:, :2], axis=0).T).sum())


def segment(df: pd.DataFrame) -> list[tuple[np.ndarray, int]]:
    """Raw chunks as ([x, y, t] array, kind). A new chunk starts after a pause or any click event."""
    click = df["state"].isin(["Pressed", "Released"]).to_numpy()
    clicks_before = np.cumsum(click)
    keep = ~click
    t = df["t"].to_numpy()[keep]
    xyt = np.column_stack([df["x"].to_numpy()[keep], df["y"].to_numpy()[keep], t])
    drag = df["state"].to_numpy()[keep] == "Drag"
    clicked = np.diff(clicks_before[keep], prepend=clicks_before[keep][0]) > 0
    new_chunk = (np.diff(t, prepend=t[0]) > config["pause_gap_s"]) | clicked
    starts = np.flatnonzero(new_chunk)
    starts = np.concatenate([[0], starts[starts > 0], [len(t)]])
    chunks = []
    for lo, hi in zip(starts[:-1], starts[1:]):
        if hi - lo >= config["min_points"] and path_length(xyt[lo:hi]) >= config["min_length_px"]:
            chunks.append((xyt[lo:hi], DRAG if drag[lo:hi].mean() > 0.5 else MOVE))
    return chunks


def normalize(chunk: np.ndarray, n: int) -> np.ndarray | None:
    """Start at the origin, bounding-box diagonal 1, n points evenly spaced by arc length."""
    xy = chunk[:, :2] - chunk[0, :2]
    diag = np.hypot(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
    arc = np.concatenate([[0.0], np.cumsum(np.hypot(*np.diff(xy, axis=0).T))])
    if diag == 0 or arc[-1] == 0:
        return None
    grid = np.linspace(0, arc[-1], n)
    return np.column_stack([np.interp(grid, arc, xy[:, 0]), np.interp(grid, arc, xy[:, 1])]) / diag


class Session:
    """n_chunks counts every recorded chunk (also ones too flat to compare); paths holds the comparable ones."""

    def __init__(self, key: str, chunks: list[tuple[np.ndarray, int]]):
        n = config["resample_n"]
        self.key, self.n_chunks, self.paths = key, len(chunks), {}
        for kind in (MOVE, DRAG):
            paths = [normalize(c, n) for c, k in chunks if k == kind]
            self.paths[kind] = np.array([p for p in paths if p is not None]).reshape(-1, n, 2)


_SESSIONS: dict[str, Session] = {}


def get_session(user: str, name: str) -> Session:
    key = f"{user}/{name}"
    if key not in _SESSIONS:
        _SESSIONS[key] = Session(key, segment(load_session(DATASET / user / name)))
    return _SESSIONS[key]


demo = get_session("user7", sorted(p.name for p in (DATASET / "user7").iterdir())[0])
print(demo.key, "->", demo.n_chunks, "chunks;", {("move", "drag")[k]: len(v) for k, v in demo.paths.items()}, "comparable")

user7/session_0041905381 -> 1583 chunks; {'move': 1428, 'drag': 155} comparable


## 2. Match chunks with banded DTW

The DTW gives up as soon as every cell of a row exceeds the tolerance, which keeps millions of comparisons fast.

In [3]:
@njit(cache=True)
def _dtw_within(a, b, band, cap, prev, cur):
    """True if the banded DTW cost between paths a and b is at most `cap`."""
    n = a.shape[0]
    prev[:] = np.inf
    prev[0] = 0.0
    for i in range(n):
        cur[:] = np.inf
        row_min = np.inf
        for j in range(max(0, i - band), min(n - 1, i + band) + 1):
            dx, dy = a[i, 0] - b[j, 0], a[i, 1] - b[j, 1]
            value = np.sqrt(dx * dx + dy * dy) + min(prev[j], prev[j + 1], cur[j])
            cur[j + 1] = value
            row_min = min(row_min, value)
        if row_min > cap:
            return False
        prev, cur = cur, prev
    return prev[n] <= cap


@njit(parallel=True, cache=True)
def has_match(queries, library, band, cap):
    """Per query path: does any library path lie within DTW cost `cap`?"""
    n = queries.shape[1]
    found = np.zeros(queries.shape[0], dtype=np.bool_)
    for q in prange(queries.shape[0]):
        prev, cur = np.empty(n + 1), np.empty(n + 1)
        for k in range(library.shape[0]):
            if _dtw_within(queries[q], library[k], band, cap, prev, cur):
                found[q] = True
                break
    return found


_PAIRS: dict = {}  # (query session, library session) -> {kind: bool per query chunk}


def pair_matches(query: Session, library: Session) -> dict[int, np.ndarray]:
    key = (query.key, library.key)
    if key not in _PAIRS:
        n = config["resample_n"]
        band, cap = int(round(config["dtw_band"] * n)), config["dtw_tolerance"] * n
        _PAIRS[key] = {
            kind: has_match(paths, library.paths[kind], band, cap) if len(library.paths[kind])
            else np.zeros(len(paths), dtype=bool)
            for kind, paths in query.paths.items()
        }
    return _PAIRS[key]


def match_ratio(query: Session, library: list[Session]) -> float:
    """Share of the query session's chunks (same kind only) that match a chunk in any library session."""
    matched = 0
    for kind, paths in query.paths.items():
        found = np.zeros(len(paths), dtype=bool)
        for session in library:
            found |= pair_matches(query, session)[kind]
        matched += int(found.sum())
    return matched / max(query.n_chunks, 1)

## 3. Draw users and sessions

The users are drawn once per trial (3 legitimate, 2 impostor, no overlap). Each user's sessions are shuffled once, and step *k* takes the first *k*,
so the 2-session profile contains the 1-session profile and the 3-session profile contains both.

In [4]:
ALL_USERS = sorted(p.name for p in DATASET.iterdir() if p.is_dir())


def draw_trial(seed: int) -> tuple[list[str], list[str], dict[str, list[str]]]:
    rng = random.Random(seed)
    legit = sorted(rng.sample(ALL_USERS, N_LEGITIMATE))
    impostor = sorted(rng.sample([u for u in ALL_USERS if u not in legit], N_IMPOSTOR))
    order = {}
    for user in legit + impostor:
        names = sorted(p.name for p in (DATASET / user).iterdir())
        random.Random(f"{seed}-{user}").shuffle(names)
        order[user] = names
    return legit, impostor, order


legit, impostor, order = draw_trial(SEED)
print("legitimate:", legit)
print("impostor:  ", impostor)
pd.DataFrame({u: pd.Series(names) for u, names in order.items()}).head(max(SESSION_STEPS))

legitimate: ['user12', 'user15', 'user21']
impostor:   ['user20', 'user9']


,user12,user15,user21,user20,user9
0,session_8872593360,session_5657866014,session_5306911480,session_0214655159,session_5155383252
1,session_9838420452,session_8848361933,session_7001496014,session_0764157160,session_0335985747
2,session_2144641057,session_6715291950,session_8505229187,session_3767642011,session_7285432516


## 4. Run one trial and record each step

For every session count, each impostor is compared with each legitimate profile separately. The impostor is matched if any single profile reaches
`match_session_ratio`.

In [5]:
def run_step(legit, impostor, order, n_sessions) -> list[dict]:
    pick = lambda user: [get_session(user, name) for name in order[user][:n_sessions]]
    profiles = {user: pick(user) for user in legit}
    rows = []
    for user in impostor:
        sessions = pick(user)
        per_profile = {p: [match_ratio(s, library) for s in sessions] for p, library in profiles.items()}
        share = {p: float(np.mean([r >= config["match_shape_ratio"] for r in ratios])) for p, ratios in per_profile.items()}
        best = max(per_profile, key=lambda p: (share[p], np.mean(per_profile[p])))
        rows.append({
            "sessions": n_sessions, "impostor": user, "best_profile": best,
            "mean_chunk_ratio": round(float(np.mean(per_profile[best])), 3),
            "sessions_matched": f"{round(share[best] * len(sessions))}/{len(sessions)}",
            "matched": share[best] >= config["match_session_ratio"],
        })
    return rows


detail = pd.DataFrame([row for k in SESSION_STEPS for row in run_step(legit, impostor, order, k)])
detail

,sessions,impostor,best_profile,mean_chunk_ratio,sessions_matched,matched
0,1,user20,user12,0.243,0/1,False
1,1,user9,user12,0.196,0/1,False
2,2,user20,user12,0.262,1/2,True
3,2,user9,user12,0.263,0/2,False
4,3,user20,user12,0.333,2/3,True
5,3,user9,user15,0.359,3/3,True


In [6]:
def threshold_table(detail: pd.DataFrame) -> pd.DataFrame:
    table = detail.groupby("sessions").agg(matched_detected=("matched", "sum"), total_impostor_user=("matched", "size"))
    table["threshold"] = table["matched_detected"] / table["total_impostor_user"]
    return table


threshold_table(detail)

,matched_detected,total_impostor_user,threshold
sessions,,,
1,0,2,0.0
2,1,2,0.5
3,2,2,1.0


## 5. Repeat over random draws

One draw of 2 impostors can only give a threshold of 0, 0.5 or 1, so the same experiment is repeated for `N_TRIALS` seeds (`SEED`, `SEED + 1`, ...).
Results are recorded in `results/session_progression_trials.csv` (every impostor, every step) and `results/session_progression.csv` (one row per step).

In [7]:
rows = []
for trial in range(N_TRIALS):
    legit_t, impostor_t, order_t = draw_trial(SEED + trial)
    for k in SESSION_STEPS:
        rows += [{"trial": trial, "seed": SEED + trial, "legitimate": ",".join(legit_t), **row}
                 for row in run_step(legit_t, impostor_t, order_t, k)]
trials = pd.DataFrame(rows)

per_trial = trials.groupby(["sessions", "trial"])["matched"].mean().rename("threshold").reset_index()
summary = per_trial.groupby("sessions")["threshold"].agg(["mean", "std"]).rename(columns={"mean": "threshold_mean", "std": "threshold_std"})
summary["matched_detected_mean"] = trials.groupby(["sessions", "trial"])["matched"].sum().groupby("sessions").mean()
summary["mean_chunk_ratio"] = trials.groupby("sessions")["mean_chunk_ratio"].mean()
summary["n_legit"], summary["n_impostor"], summary["n_trials"] = N_LEGITIMATE, N_IMPOSTOR, N_TRIALS
summary = summary.round(3)

(ROOT / "results").mkdir(exist_ok=True)
trials.to_csv(ROOT / "results" / "session_progression_trials.csv", index=False)
summary.to_csv(ROOT / "results" / "session_progression.csv")
summary

,threshold_mean,threshold_std,matched_detected_mean,mean_chunk_ratio,n_legit,n_impostor,n_trials
sessions,,,,,,,
1,0.075,0.183,0.15,0.148,3,2,20
2,0.250,0.256,0.50,0.196,3,2,20
3,0.275,0.302,0.55,0.234,3,2,20


In [8]:
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.errorbar(summary.index, summary["threshold_mean"], yerr=summary["threshold_std"], marker="o", capsize=4)
ax.set_xticks(SESSION_STEPS)
ax.set_ylim(0, 1)
ax.set_xlabel("sessions per user")
ax.set_ylabel("threshold (impostors matched / total, 0 = best)")
ax.set_title(f"{N_LEGITIMATE} legitimate + {N_IMPOSTOR} impostor users, {N_TRIALS} draws")
fig.tight_layout()
(ROOT / "figures").mkdir(exist_ok=True)
fig.savefig(ROOT / "figures" / "session_progression.png", dpi=150)
plt.show()

<Figure size 550x400 with 1 Axes>

## Notes

- The tolerance in `config.yaml` (0.02) is the value that keeps the metric from being all 0 or all 1. In the earlier sweep a user's own held-out sessions never matched
  more than impostors did, so read these numbers as a baseline for how the threshold grows with more sessions, not as proof that shapes are unique to a user.
- Balabit has 10 users, so 3 + 2 users is far from the limit; the earlier script runs capped at 10 total users.
- The `src/` scripts remain the reference for the file-based flow (`temp/`, `shapes/`, `results/trial_k.json`).